# Music Generation I

## Exercise 1 [Markov Models, 4 points]

In [96]:
import pathlib
from dataclasses import dataclass
from datetime import datetime
from typing import List, Optional, Protocol, Tuple

import music21
import torch
import torch.nn.utils as F
from music21 import converter
from pomegranate.markov_chain import MarkovChain

In [97]:
current_dir = pathlib.Path.cwd()
data_path = current_dir.joinpath("data")
output_path = data_path.joinpath("output")

# File extensions
ABC_FILE_EXTENSION = ".abc"
MUSIC_XML_FILE_EXTENSION = ".xml"

# Data
STYLES = ["french", "irish_folk"]
ORDERS = [1, 2, 4]

# Constants
N_BARS = 16

In [98]:

@dataclass(frozen=True)
class State:
    note: int
    duration: int

    #score_position: Optional[int] = None
    #bar_position: Optional[int] = None
    #articulation: Optional[int] = None
    # add fields as you discover them

    def to_music21(self):
        if self.note == 0:
            return music21.note.Rest(quarterLength=self.duration)
        else:
            return music21.note.Note(self.note, quarterLength=self.duration)


class Embedding(Protocol):
    def initialize_embedding(self, states: list[State]) -> None:
        """ Initializes embedding. """

    def encode(self, state: State) -> int:
        """Project a State to a discrete token."""

    def decode(self, idx: int) -> State:
        """Optional: reconstruct a State (partial is acceptable)."""

    def get_eos_idx(self) -> int:
        """Index used for end of sentence and padding."""

    def __len__(self) -> int:
        """Vocabulary size."""


class DistinctStateEmbedding(Embedding):
    """
    One embedding for each distinct symbol.
    """

    def __init__(self):
        self.state2idx = None
        self.idx2state = None
        self.eos_idx = 0

    def initialize_embedding(self, states: list[State]):
        self.state2idx = {s: i for i, s in enumerate(states)}
        self.idx2state = list(states)
        self.eos_idx = len(self.idx2state)

    def encode(self, state: State) -> int:
        return self.state2idx[state]

    def decode(self, idx: int) -> Optional[State]:
        if idx == self.eos_idx:
            return None
        assert idx < len(self.idx2state), f"idx out of range (idx {idx} > {len(self.idx2state)} vocab_size)"
        return self.idx2state[idx]

    def get_eos_idx(self) -> int:
        return self.eos_idx

    def __len__(self) -> int:
        return len(self.idx2state)


In [99]:
def get_state_seq(file: pathlib.Path) -> list[State]:
    """
    For now only storing pitch and duration of the notes and rests
    :param file:
    :return:
    """
    states = []

    score = converter.parse(file)
    for el in score.recurse():  # TODO: this logic could be in the state...
        if isinstance(el, music21.note.Note):
            states.append(State(note=el.pitch.midi, duration=el.duration.quarterLength))
        elif isinstance(el, music21.note.Rest):
            states.append(State(note=0, duration=el.duration.quarterLength))
        else:
            continue

    return states


def get_samples(style_dir: pathlib.Path, embedding_class: type[Embedding]) -> Tuple[torch.Tensor, Embedding]:
    # Get state sequences
    state_sequences: list[list[State]] = []
    all_states: list[State] = []
    for file in sorted(style_dir.glob(f"*{ABC_FILE_EXTENSION}")):
        state_seq = get_state_seq(file)
        state_sequences.append(state_seq)
        all_states.extend(state_seq)

    # Initialize embedding
    embedding = embedding_class()
    embedding.initialize_embedding(all_states)

    # Get tensor
    tensor_list = [torch.tensor([embedding.encode(el) for el in seq]) for seq in state_sequences]
    X = F.rnn.pad_sequence(tensor_list, batch_first=True, padding_value=embedding.get_eos_idx())
    X = X.unsqueeze(-1)
    return X, embedding


def create_and_fit_markov_model(data: torch.Tensor, order) -> MarkovChain:
    model = MarkovChain(k=order)
    model.fit(data)
    return model


def sample_from_probs(probs: torch.Tensor, generator: torch.Generator) -> int:
    """
    probs: 1D tensor summing to 1
    """
    return torch.multinomial(probs, num_samples=1, generator=generator).item()


def get_initial_state(n_samples: int, vocab_size: int):  # TODO: improve this !!
    """
    for now only 0s..
    :param n_samples:
    :param vocab_size:
    :return:
    """
    return [0 for _ in range(n_samples)]


def generate_seq(model: MarkovChain, n_samples: int, vocab_size: int, seed: int) -> list[int]:
    seq = get_initial_state(n_samples=model.k, vocab_size=vocab_size)

    if seed is not None:
        gen = torch.Generator()
        gen.manual_seed(seed)
    else:
        gen = None

    for i in range(n_samples):
        dist = model.distributions[model.k].probs[0]
        context = seq[-model.k:]
        probs_i = dist[tuple(context)]
        i_idx = sample_from_probs(probs_i, generator=gen)
        seq.append(i_idx)

    return seq


def get_music_xml_str(idx_seq: List[int], embedding: Embedding) -> str:
    state_seq = [s for s in [embedding.decode(x) for x in idx_seq] if s is not None]
    #print(state_seq)

    stream = music21.stream.Stream()
    for state in state_seq:
        stream.append(state.to_music21())

    score = music21.stream.Score()
    part = music21.stream.Part()
    part.append(stream.makeMeasures())
    score.append(part)

    score_tmp_path = pathlib.Path(score.write('musicxml'))
    music_xml_str = score_tmp_path.read_text()
    return music_xml_str


In [100]:
def get_seqs(styles: List[str], orders: List[int], embedding_class: Optional[type[Embedding]] = DistinctStateEmbedding,
             verbose=False, n_samples=N_BARS,
             seed: Optional[int] = 42) -> List[pathlib.Path]:
    gen_files = []
    for style in styles:
        if verbose:
            print(f"\nProcessing -> {style}")

        data, embedding = get_samples(data_path.joinpath(style), embedding_class)
        if verbose:
            print(f"Vocabulary size: {len(embedding)}")
            print(f"data shape: {data.shape}")

        for order in orders:
            if verbose:
                print(f"{style} - {order}")

            if verbose:
                print("Training model")
            model = create_and_fit_markov_model(data=data, order=order)
            if verbose:
                print("Done!")
            idx_seq = generate_seq(model, n_samples=n_samples, vocab_size=len(embedding), seed=seed)

            if verbose:
                print(f"{idx_seq}")

            music_xml_str = get_music_xml_str(idx_seq, embedding=embedding)

            #print(music_xml_str)
            gen_files.append(music_xml_str)

            output_file_name = f"{datetime.now().strftime('%y%m%d-%H%M%S')}--{style}-{order}{MUSIC_XML_FILE_EXTENSION}"
            with open(output_path.joinpath(output_file_name), "w+") as f:
                f.write(music_xml_str)

    return gen_files


In [101]:
# Demo
_ = get_seqs(["french"], orders=[1, 2], verbose=True, n_samples=60)


Processing -> french
Vocabulary size: 263
data shape: torch.Size([5, 69, 1])
french - 1
Training model
Done!
[0, 211, 33, 36, 30, 240, 114, 116, 154, 129, 211, 15, 217, 210, 217, 212, 213, 138, 242, 241, 261, 260, 261, 262, 251, 53, 54, 221, 217, 40, 253, 254, 253, 254, 259, 258, 260, 261, 260, 260, 261, 261, 260, 249, 249, 217, 109, 176, 217, 251, 251, 216, 141, 257, 258, 259, 258, 259, 258, 257, 14]
french - 2
Training model
Done!
[0, 0, 211, 33, 36, 30, 240, 114, 116, 154, 129, 211, 15, 217, 116, 247, 156, 112, 241, 247, 114, 24, 195, 195, 217, 88, 164, 162, 123, 152, 61, 209, 126, 211, 71, 132, 253, 12, 81, 152, 88, 83, 112, 96, 55, 58, 155, 234, 84, 171, 74, 229, 77, 98, 17, 261, 71, 51, 211, 132, 152, 24]


In [102]:
#get_seqs(STYLES, ORDERS)